In [1]:
import pandas as pd

In [2]:
# 0) 데이터 로드
detail_path = "../data/store_review_detail(new_전처리중).csv"
summary_path = "../data/store_review_summary(전처리중).csv"

In [3]:
df_detail = pd.read_csv(detail_path)
df_summary = pd.read_csv(summary_path)

In [4]:
# 1) 컬럼명 확인 (가독성용)
detail_store_name_col = "검색한매장명(store_id 매치용 임시)"
detail_raw_visit_col = "방문 일자(visit_date), 방문 횟수(visit_count)"
detail_visit_time_col = "visit_time"

summary_store_name_col = "store_name(임시)"
summary_store_id_col = "store_id"

In [5]:
# 2) store_id 매핑 딕셔너리 생성
store_map = (
    df_summary[[summary_store_name_col, summary_store_id_col]]
    .drop_duplicates()
    .set_index(summary_store_name_col)[summary_store_id_col]
    .to_dict()
)

In [6]:
# 3) store_id 채우기 (기존 컬럼 새로 정리)
df_detail["store_id"] = df_detail[detail_store_name_col].map(store_map)

# 정수형(결측 허용)으로 변환
df_detail["store_id"] = pd.to_numeric(df_detail["store_id"], errors="coerce").astype("Int64")

df_detail.head(3)

,store_id(*채워야함),검색한매장명(store_id 매치용 임시),reviewer_id,"방문 일자(visit_date), 방문 횟수(visit_count)",visit_time,ok,error,store_id
0,1509.0,역삼아레나빌딩,어쩌라고4036,방문일 2.6.금 2026년 2월 6일 금요일 1번째 방문 인증 수단 영수증,아침,True,NaN,1509
1,NaN,역삼아레나빌딩,Wonderfulworld,방문일 1.14.수 2026년 1월 14일 수요일 8번째 방문 인증 수단 영수증,점심,True,NaN,1509
2,NaN,역삼아레나빌딩,행복한미자씨,방문일 1.10.토 2026년 1월 10일 토요일 16번째 방문 인증 수단 영수증,아침,True,NaN,1509


In [7]:
# 4) visit_date 추출 (문자열)
# 예: "2026년 2월 6일"
df_detail["visit_date_raw"] = df_detail[detail_raw_visit_col].str.extract(
    r"(\d{4}년\s*\d{1,2}월\s*\d{1,2}일)"
)
df_detail.head(3)

,store_id(*채워야함),검색한매장명(store_id 매치용 임시),reviewer_id,"방문 일자(visit_date), 방문 횟수(visit_count)",visit_time,ok,error,store_id,visit_date_raw
0,1509.0,역삼아레나빌딩,어쩌라고4036,방문일 2.6.금 2026년 2월 6일 금요일 1번째 방문 인증 수단 영수증,아침,True,NaN,1509,2026년 2월 6일
1,NaN,역삼아레나빌딩,Wonderfulworld,방문일 1.14.수 2026년 1월 14일 수요일 8번째 방문 인증 수단 영수증,점심,True,NaN,1509,2026년 1월 14일
2,NaN,역삼아레나빌딩,행복한미자씨,방문일 1.10.토 2026년 1월 10일 토요일 16번째 방문 인증 수단 영수증,아침,True,NaN,1509,2026년 1월 10일


In [8]:
# 5) visit_count 추출 (숫자)
# 예: "1번째 방문" -> 1
df_detail["visit_count"] = df_detail[detail_raw_visit_col].str.extract(
    r"(\d+)번째\s*방문"
)

df_detail["visit_count"] = pd.to_numeric(df_detail["visit_count"], errors="coerce").astype("Int64")
df_detail.head(3)

,store_id(*채워야함),검색한매장명(store_id 매치용 임시),reviewer_id,"방문 일자(visit_date), 방문 횟수(visit_count)",visit_time,ok,error,store_id,visit_date_raw,visit_count
0,1509.0,역삼아레나빌딩,어쩌라고4036,방문일 2.6.금 2026년 2월 6일 금요일 1번째 방문 인증 수단 영수증,아침,True,NaN,1509,2026년 2월 6일,1
1,NaN,역삼아레나빌딩,Wonderfulworld,방문일 1.14.수 2026년 1월 14일 수요일 8번째 방문 인증 수단 영수증,점심,True,NaN,1509,2026년 1월 14일,8
2,NaN,역삼아레나빌딩,행복한미자씨,방문일 1.10.토 2026년 1월 10일 토요일 16번째 방문 인증 수단 영수증,아침,True,NaN,1509,2026년 1월 10일,16


In [9]:
# 6) visit_date를 datetime으로 변환
# 예: "2026년 2월 6일" -> "2026-02-06"
df_detail["visit_date"] = (
    df_detail["visit_date_raw"]
    .str.replace("년", "-", regex=False)
    .str.replace("월", "-", regex=False)
    .str.replace("일", "", regex=False)
    .str.replace(" ", "", regex=False)
)

df_detail["visit_date"] = pd.to_datetime(df_detail["visit_date"], errors="coerce")
df_detail.head(3)

,store_id(*채워야함),검색한매장명(store_id 매치용 임시),reviewer_id,"방문 일자(visit_date), 방문 횟수(visit_count)",visit_time,ok,error,store_id,visit_date_raw,visit_count,visit_date
0,1509.0,역삼아레나빌딩,어쩌라고4036,방문일 2.6.금 2026년 2월 6일 금요일 1번째 방문 인증 수단 영수증,아침,True,NaN,1509,2026년 2월 6일,1,2026-02-06
1,NaN,역삼아레나빌딩,Wonderfulworld,방문일 1.14.수 2026년 1월 14일 수요일 8번째 방문 인증 수단 영수증,점심,True,NaN,1509,2026년 1월 14일,8,2026-01-14
2,NaN,역삼아레나빌딩,행복한미자씨,방문일 1.10.토 2026년 1월 10일 토요일 16번째 방문 인증 수단 영수증,아침,True,NaN,1509,2026년 1월 10일,16,2026-01-10


In [10]:
# 7) 필요한 컬럼만 우선 추리기 (보류 포함)
final_cols = [
    "store_id",
    "reviewer_id",
    "visit_date",
    "visit_count",
    "visit_time",
]
final_cols

['store_id', 'reviewer_id', 'visit_date', 'visit_count', 'visit_time']

In [11]:
# 8) 검증 출력
print("=== 전처리 결과 미리보기 ===")
df_out = df_detail[final_cols]
df_out.head(3)

=== 전처리 결과 미리보기 ===


,store_id,reviewer_id,visit_date,visit_count,visit_time
0,1509,어쩌라고4036,2026-02-06,1,아침
1,1509,Wonderfulworld,2026-01-14,8,점심
2,1509,행복한미자씨,2026-01-10,16,아침


In [12]:
# 9) 결측 현황 확인
print("\n=== 결측치 개수 ===")
print(df_out[["store_id", "reviewer_id", "visit_date", "visit_count", "visit_time"]].isna().sum())


=== 결측치 개수 ===
store_id        86
reviewer_id     21
visit_date     108
visit_count    108
visit_time       0
dtype: int64


In [13]:
# 10) 결측 행 제거
df_out_clean = df_out.dropna(subset=["store_id", "reviewer_id", "visit_date", "visit_count", "visit_time"]).copy()

print("\n=== 결측치 개수 ===")
print(df_out_clean[["store_id", "reviewer_id", "visit_date", "visit_count", "visit_time"]].isna().sum())


=== 결측치 개수 ===
store_id       0
reviewer_id    0
visit_date     0
visit_count    0
visit_time     0
dtype: int64


In [14]:
# 11) 저장
out_path = "../data/store_review_detail.csv"
df_out_clean.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {out_path}")


저장 완료: ../data/store_review_detail.csv
